# Retrieval-Augmented Generation Exercise

In this exercise, I compare three approaches:

1. Asking the LLM directly without article context
2. Manual RAG using FAISS and sentence-transformer embeddings
3. LangChain RAG using retriever and generator components

The goal is to see whether adding article context helps the model give a more accurate answer about SC-LoRA.

In [1]:
# Import Required Libraries

from openai import OpenAI
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
import numpy as np

print("Imports successful!")

Imports successful!


In [2]:
# Load the SC-LoRA Article

with open("data/2505.23724v1.txt", "r", encoding="utf-8") as fi:
    article = fi.read()

print(len(article))

49755


In [3]:
article[:1000]

'arXiv:2505.23724v1  [cs.LG]  29 May 2025SC-LoRA: Balancing Efficient Fine-tuning and Knowledge Preservation via\nSubspace-Constrained LoRA\nMinrui Luo∗1,2, Fuhang Kuang∗1, Yu Wang3, Zirui Liu1, Tianxing He†1,2\n1Institute for Interdisciplinary Information Sciences, Tsinghua University\n2Shanghai Qi Zhi Institute\n3Institute of Information Engineering, Chinese Academy of Sciences\n{luomr22,kfh22,liu-zr22}@mails.tsinghua.edu.cn;\nwangyu2002@iie.ac.cn hetianxing@mail.tsinghua.edu.cn\nAbstract\nParameter-Efficient Fine-Tuning (PEFT)\nmethods, particularly Low-Rank Adaptation\n(LoRA), are indispensable for efficiently\ncustomizing Large Language Models (LLMs).\nHowever, vanilla LoRA suffers from slow\nconvergence speed and knowledge forgetting\nproblems. Recent studies have leveraged\nthe power of designed LoRA initialization,\nto enhance the fine-tuning efficiency, or to\npreserve knowledge in the pre-trained LLM.\nHowever, none of these works can address\nthe two cases at the same time. 

In [22]:
# Set Up OpenRouter Client
import getpass

api_key = getpass.getpass(": ")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

model_name = "openrouter/owl-alpha"

:  ········


In [23]:
# Test OpenRouter Connection
response = client.chat.completions.create(
    model=model_name,
    messages=[
        {
            "role": "user",
            "content": "Hello"
        }
    ]
)

print(response.choices[0].message.content)

Hello! How can I help you today?


In [25]:
# Ask LLM Without Context
query = "How does SC-LoRA differ from regular LoRA?"

response = client.chat.completions.create(
    model=model_name,
    messages=[
        {
            "role": "user",
            "content": query
        }
    ]
)

print(response.choices[0].message.content)

SC-LoRA modifies LoRA by introducing task-specific sub-networks and constraints on the selection of updates learned for each task. While LoRA fine-tunes the low-rank updates without any restrictions, SC-LoRA's approach allows for more focused updates that are tailored to each individual task, leading to better performance in multi-task sequential finetuning.


The answer appears to be mostly a guess. The model incorrectly defined SC-LoRA and provided an explanation that sounded reasonable but was not supported by the article. This demonstrates a limitation of using an LLM without retrieval, especially for recently published research papers.

In [7]:
# Manual RAG - Split Article into Chunks
# The article is too long to search as one large text.
# We split it into smaller chunks so FAISS can later
# find the most relevant parts of the article.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_text(article)

print("Number of chunks:", len(chunks))

Number of chunks: 112


In [8]:
chunks[0]

'arXiv:2505.23724v1  [cs.LG]  29 May 2025SC-LoRA: Balancing Efficient Fine-tuning and Knowledge Preservation via\nSubspace-Constrained LoRA\nMinrui Luo∗1,2, Fuhang Kuang∗1, Yu Wang3, Zirui Liu1, Tianxing He†1,2\n1Institute for Interdisciplinary Information Sciences, Tsinghua University\n2Shanghai Qi Zhi Institute\n3Institute of Information Engineering, Chinese Academy of Sciences\n{luomr22,kfh22,liu-zr22}@mails.tsinghua.edu.cn;\nwangyu2002@iie.ac.cn hetianxing@mail.tsinghua.edu.cn\nAbstract'

In [9]:
# Load Embedding Model
# This model converts text into embeddings (vectors)

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [10]:
# Create Embeddings for Article Chunks
# Each chunk will be converted into a vector of numbers.

chunk_embeddings = embedding_model.encode(chunks)

print(chunk_embeddings.shape)

(112, 384)


In [11]:
# Create FAISS Index
# FAISS stores the embeddings so we can search them.

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(chunk_embeddings)

print("Number of vectors in FAISS index:", index.ntotal)

Number of vectors in FAISS index: 112


In [12]:
# Create Query Embedding
# Convert the question into the same kind of vector
# as the article chunks.

query_embedding = embedding_model.encode([query])

print(query_embedding.shape)

(1, 384)


In [13]:
# Search FAISS for Relevant Chunks
# k = number of chunks we want to retrieve

k = 3

distances, indices = index.search(query_embedding, k)

print("Distances:", distances)
print("Indices:", indices)

Distances: [[0.9403492  0.97954905 1.0226405 ]]
Indices: [[50 44 39]]


In [14]:
# View Retrieved Chunks
for i in indices[0]:
    print("=" * 80)
    print(f"Chunk {i}")
    print(chunks[i])

Chunk 50
methods, both in utility and safety metric. Com-
pared to the original model, SC-LoRA ( β= 0.9)
exhibits almost no safety degradation, and achieves
best utility, even surpassing full fine-tuning by 3.79
points. When increasing the learning rate, LoRA
shows a sharp decline in safety alignment while
math ability is increasing. LoRA (lr=2e-5) and
CorDA KPA, though preserving safety well, are
insufficient in fine-tuning performance compared
to our method. PiSSA and CorDA IPA, though
Chunk 44
sponses (score = 5) as harmfulness rate . Lower
values for both metrics indicate stronger safety of
the model.
5Method #Params HS↓HR(%) ↓Utility ↑
Llama-2-7b-Chat - 1.100 1.212 24.13
Full fine-tuning 6738M 1.364 5.455 51.41
LoRA 320M 1.176 2.424 50.32
PiSSA 320M 1.252 4.242 51.87
CorDA IPA 320M 1.209 3.333 44.61
CorDA KPA 320M 1.106 0.606 50.89
SC-LoRAβ= 0.5 320M 1.161 1.818 52.54
β= 0.7 320M 1.148 1.818 52.07
β= 0.9 320M 1.097 0.000 51.67
Chunk 39
2019) with the following hyper-parameters: ba

In [15]:
# Retrieve More Chunks
k = 5

distances, indices = index.search(
    query_embedding,
    k
)

print(indices)

[[50 44 39 65 55]]


In [16]:
for i in indices[0]:
    print("=" * 80)
    print(f"Chunk {i}")
    print(chunks[i])

Chunk 50
methods, both in utility and safety metric. Com-
pared to the original model, SC-LoRA ( β= 0.9)
exhibits almost no safety degradation, and achieves
best utility, even surpassing full fine-tuning by 3.79
points. When increasing the learning rate, LoRA
shows a sharp decline in safety alignment while
math ability is increasing. LoRA (lr=2e-5) and
CorDA KPA, though preserving safety well, are
insufficient in fine-tuning performance compared
to our method. PiSSA and CorDA IPA, though
Chunk 44
sponses (score = 5) as harmfulness rate . Lower
values for both metrics indicate stronger safety of
the model.
5Method #Params HS↓HR(%) ↓Utility ↑
Llama-2-7b-Chat - 1.100 1.212 24.13
Full fine-tuning 6738M 1.364 5.455 51.41
LoRA 320M 1.176 2.424 50.32
PiSSA 320M 1.252 4.242 51.87
CorDA IPA 320M 1.209 3.333 44.61
CorDA KPA 320M 1.106 0.606 50.89
SC-LoRAβ= 0.5 320M 1.161 1.818 52.54
β= 0.7 320M 1.148 1.818 52.07
β= 0.9 320M 1.097 0.000 51.67
Chunk 39
2019) with the following hyper-parameters: ba

In [17]:
# Combine the retrieved chunks into one context string
# that will be passed to the LLM.

context = "\n\n".join([chunks[i] for i in indices[0]])

print(context[:1500])

methods, both in utility and safety metric. Com-
pared to the original model, SC-LoRA ( β= 0.9)
exhibits almost no safety degradation, and achieves
best utility, even surpassing full fine-tuning by 3.79
points. When increasing the learning rate, LoRA
shows a sharp decline in safety alignment while
math ability is increasing. LoRA (lr=2e-5) and
CorDA KPA, though preserving safety well, are
insufficient in fine-tuning performance compared
to our method. PiSSA and CorDA IPA, though

sponses (score = 5) as harmfulness rate . Lower
values for both metrics indicate stronger safety of
the model.
5Method #Params HS↓HR(%) ↓Utility ↑
Llama-2-7b-Chat - 1.100 1.212 24.13
Full fine-tuning 6738M 1.364 5.455 51.41
LoRA 320M 1.176 2.424 50.32
PiSSA 320M 1.252 4.242 51.87
CorDA IPA 320M 1.209 3.333 44.61
CorDA KPA 320M 1.106 0.606 50.89
SC-LoRAβ= 0.5 320M 1.161 1.818 52.54
β= 0.7 320M 1.148 1.818 52.07
β= 0.9 320M 1.097 0.000 51.67

2019) with the following hyper-parameters: batch
size 128, learning ra

In [20]:
# Ask LLM With Retrieved Context
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentences maximum and keep the answer concise. "
    f"Context: {context}"
)

response = client.chat.completions.create(
    model=model_name,
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": query
        }
    ]
)

print(response.choices[0].message.content)

SC-LoRA is a LoRA initialization method that focuses on preserving knowledge during fine-tuning, whereas regular LoRA does not prioritize this. SC-LoRA achieves better utility and safety metrics compared to regular LoRA, which shows a sharp decline in safety alignment when the learning rate is increased. However, SC-LoRA does not strongly constrain updates during the fine-tuning process, which can lead to a drop in knowledge preservation ability over more complex tasks.


## Part 1 = Build RAG yourself
## Part 2 = Let LangChain build the same RAG pipeline for you

In [26]:
from langchain_community.vectorstores import FAISS

In [27]:
# Create Embedding Model
# LangChain wrapper around all-MiniLM-L6-v2

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings ready!


In [28]:
# Create FAISS Vector Store
# LangChain will create embeddings and store them
# in FAISS automatically.

vectorstore = FAISS.from_texts(
    texts=chunks,
    embedding=embeddings
)

print("Vector store created!")

Vector store created!


In [29]:
# Create Retriever
# The retriever will search the vector store
# for relevant chunks.

retriever = vectorstore.as_retriever()

print("Retriever created!")

Retriever created!


In [30]:
# Create LLM
# LangChain wrapper around OpenRouter

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    model_name="openrouter/owl-alpha",
    openai_api_key=api_key
)

print("LLM created!")

LLM created!


In [31]:
# Create Prompt Template
system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}")
    ]
)

print("Prompt created!")

Prompt created!


In [32]:
# Format Retrieved Documents
# Combine retrieved chunks into one context string

def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

print("Helper function created!")

Helper function created!


In [33]:
# Create RAG Chain
rag_chain = (
    {
        "context": (
            RunnableLambda(
                lambda x: x["question"]
            )
            | retriever
            | RunnableLambda(format_docs)
        ),
        "question": RunnableLambda(
            lambda x: x["question"]
        )
    }
    | prompt
    | llm
)

print("RAG chain created!")

RAG chain created!


In [34]:
# Invoke RAG Chain
response = rag_chain.invoke(
    {"question": query}
)

print(response.content)

SC-LoRA differs from regular LoRA primarily through its initialization method and hyperparameter tuning. While regular LoRA uses standard initialization, SC-LoRA employs a specialized initialization that better preserves the original model's knowledge and safety alignment. The key distinction is that SC-LoRA tunes a hyperparameter β to achieve an optimal balance between utility and safety, whereas regular LoRA lacks this specific optimization mechanism.


The LLM-only response was inaccurate because it incorrectly expanded SC-LoRA as "Stochastic Contextual Low-Rank Adaptation" and appeared to generate a plausible explanation without using information from the paper.

Adding retrieval through Manual RAG improved the response by providing relevant context from the article. The answer was more grounded in the paper and avoided the hallucinated definition. However, the quality of the response still depended on the relevance of the retrieved chunks.

The LangChain RAG output was similar to the Manual RAG output because both approaches used the same article, embedding model, retriever, and language model. LangChain did not improve the model itself; instead, it simplified the retrieval and generation workflow by automating the retrieval, context formatting, prompt creation, and LLM interaction steps.